# BERT TRAINING 

In [6]:
import pandas as pd
import numpy as np
import os
import torch
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from datasets import Dataset
from transformers import (
    DistilBertTokenizerFast,
    DistilBertForSequenceClassification,
    TrainingArguments,
    Trainer,
)
import joblib

print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

# 1) LOAD DATA
csv_path = "augmented_ready.csv"   
df = pd.read_csv(csv_path)
print("Loaded:", csv_path, "Shape:", df.shape)

# Use CLEAN_TEXT as input and CATEGORY as label
df["text"] = df["CLEAN_TEXT"].astype(str)
df["label"] = df["CATEGORY"].astype(str)

print("\nSample rows:")
print(df[["text", "label"]].head())

print("\nLabel value counts:")
print(df["label"].value_counts())

# 2) ENCODE LABELS
le = LabelEncoder()
df["label_id"] = le.fit_transform(df["label"])

print("\nLabel classes mapped to IDs:")
for i, cls in enumerate(le.classes_):
    print(f"{i} → {cls}")

num_labels = len(le.classes_)
os.makedirs("bert_finetuned", exist_ok=True)
joblib.dump(le, "bert_finetuned/label_encoder.pkl")
print("\nSaved label encoder to bert_finetuned/label_encoder.pkl")

# 3) TRAIN / TEST SPLIT
train_df, test_df = train_test_split(
    df[["text", "label_id"]],
    test_size=0.2,
    stratify=df["label_id"],
    random_state=42,
)

print("\nTrain shape:", train_df.shape, "Test shape:", test_df.shape)

# 4) CONVERT TO HF DATASETS
train_ds = Dataset.from_pandas(train_df.reset_index(drop=True))
test_ds  = Dataset.from_pandas(test_df.reset_index(drop=True))

# 5) TOKENIZER
tokenizer = DistilBertTokenizerFast.from_pretrained("distilbert-base-uncased")

def tokenize_batch(batch):
    return tokenizer(
        batch["text"],
        padding="max_length",
        truncation=True,
        max_length=64,
    )

train_enc = train_ds.map(tokenize_batch, batched=True)
test_enc  = test_ds.map(tokenize_batch, batched=True)

train_enc = train_enc.rename_column("label_id", "labels")
test_enc  = test_enc.rename_column("label_id", "labels")

train_enc.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])
test_enc.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

# 6) MODEL
model = DistilBertForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=num_labels
)

# 7) METRICS
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    macro_f1 = f1_score(labels, preds, average="macro")
    return {"macro_f1": macro_f1}

# 8) TRAINING ARGS  
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="./bert_results",
    num_train_epochs=3,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=128,
    learning_rate=2e-5,
    logging_dir="./logs",
    logging_steps=200,
    save_total_limit=2,
  
)

# 9) TRAINER
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_enc,
    eval_dataset=test_enc,
    compute_metrics=compute_metrics,
)

print("\nStarting training…")
trainer.train()

print("\nEvaluating on test set…")
res = trainer.evaluate(test_enc)
print("Evaluation metrics:", res)

# 10) DETAILED METRICS
from sklearn.metrics import classification_report, confusion_matrix

pred_out = trainer.predict(test_enc)
logits = pred_out.predictions
y_true = pred_out.label_ids
y_pred = np.argmax(logits, axis=1)

print("\nClassification report:")
print(classification_report(
    y_true,
    y_pred,
    target_names=list(le.classes_),
    zero_division=0
))

cm = confusion_matrix(y_true, y_pred)
print("\nConfusion matrix:\n", cm)

# 11) SAVE MODEL + TOKENIZER
trainer.save_model("bert_finetuned")
tokenizer.save_pretrained("bert_finetuned")
print("\nSaved fine-tuned model & tokenizer to ./bert_finetuned")

# 12) OPTIONAL: EXPORT TEST PREDICTIONS
test_texts = test_df["text"].tolist()
test_pred_labels = le.inverse_transform(y_pred)

test_export = pd.DataFrame({
    "text": test_texts,
    "true_label": le.inverse_transform(y_true),
    "pred_label": test_pred_labels,
})
test_export.to_csv("bert_test_predictions.csv", index=False)
print("Exported test predictions to bert_test_predictions.csv")



Torch version: 2.5.1
CUDA available: True
Loaded: augmented_ready.csv Shape: (41068, 30)

Sample rows:
                            text     label
0       fraud bedi krish pvt ltd     Other
1  neft fraud bedi krish pvt ltd  Transfer
2   fraud bedi krish pvt ltd mkt  Shopping
3       fraud bedi krish pvt ltd     Other
4                            nan     Other

Label value counts:
label
Other              30664
Shopping            5681
Transfer            3585
Cash Withdrawal     1138
Name: count, dtype: int64

Label classes mapped to IDs:
0 → Cash Withdrawal
1 → Other
2 → Shopping
3 → Transfer

Saved label encoder to bert_finetuned/label_encoder.pkl

Train shape: (32854, 2) Test shape: (8214, 2)


Map:   0%|          | 0/32854 [00:00<?, ? examples/s]

Map:   0%|          | 0/8214 [00:00<?, ? examples/s]

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



Starting training…


Step,Training Loss
200,0.266700
400,0.004200
600,0.001600
800,0.000900
1000,0.000600
1200,0.000400
1400,0.000400
1600,0.000300
1800,0.000200
2000,0.000200



Evaluating on test set…


Evaluation metrics: {'eval_loss': 7.484776142518967e-05, 'eval_macro_f1': 1.0, 'eval_runtime': 58.8687, 'eval_samples_per_second': 139.531, 'eval_steps_per_second': 1.104, 'epoch': 3.0}

Classification report:
                 precision    recall  f1-score   support

Cash Withdrawal       1.00      1.00      1.00       228
          Other       1.00      1.00      1.00      6133
       Shopping       1.00      1.00      1.00      1136
       Transfer       1.00      1.00      1.00       717

       accuracy                           1.00      8214
      macro avg       1.00      1.00      1.00      8214
   weighted avg       1.00      1.00      1.00      8214


Confusion matrix:
 [[ 228    0    0    0]
 [   0 6133    0    0]
 [   0    0 1136    0]
 [   0    0    0  717]]

Saved fine-tuned model & tokenizer to ./bert_finetuned
Exported test predictions to bert_test_predictions.csv


In [7]:
import os
print(os.listdir("bert_finetuned"))


['config.json', 'label_encoder.pkl', 'model.safetensors', 'special_tokens_map.json', 'tokenizer.json', 'tokenizer_config.json', 'training_args.bin', 'vocab.txt']


In [8]:
model = DistilBertForSequenceClassification.from_pretrained("bert_finetuned")
tokenizer = DistilBertTokenizerFast.from_pretrained("bert_finetuned")
le = joblib.load("bert_finetuned/label_encoder.pkl")


In [19]:
import requests

tests = [
    "upi flipkart mkt#332 499.00",
    "atm icici chennai 2000.00",
    "fraud_Random Pvt Ltd 3999.99",
]

for t in tests:
    r = requests.post("http://127.0.0.1:8000/predict", json={"text": t})
    print("\nTEXT:", t)
    print(r.json())



TEXT: upi flipkart mkt#332 499.00
{'input': 'upi flipkart mkt#332 499.00', 'final_label': 'Shopping', 'final_confidence': 0.999855637550354, 'source': 'bert', 'bert_confidence': 0.999855637550354, 'probs': [5.2009625505888835e-05, 3.134265716653317e-05, 0.999855637550354, 6.105159991420805e-05]}

TEXT: atm icici chennai 2000.00
{'input': 'atm icici chennai 2000.00', 'final_label': 'Cash Withdrawal', 'final_confidence': 0.9990234375, 'source': 'bert', 'bert_confidence': 0.9990234375, 'probs': [0.9990234375, 0.00015140490722842515, 0.0005509782931767404, 0.0002742286887951195]}

TEXT: fraud_Random Pvt Ltd 3999.99
{'input': 'fraud_Random Pvt Ltd 3999.99', 'final_label': 'Other', 'final_confidence': 0.9208264350891113, 'source': 'bert', 'bert_confidence': 0.9208264350891113, 'probs': [0.0005095354281365871, 0.9208264350891113, 0.07657159864902496, 0.0020924420095980167]}


In [4]:
import os
print("Exists?", os.path.exists("streamlit_app.py"))


Exists? True


In [16]:
import os
print("streamlit_app.py exists?", os.path.exists("streamlit_app.py"))
print("Working dir:", os.getcwd())
print("Sample CSV exists?", os.path.exists("augmented_ready.csv"))


streamlit_app.py exists? True
Working dir: C:\Users\bhuvi
Sample CSV exists? True


# UI - STREAMLIT 

In [10]:
%%writefile streamlit_app.py
import streamlit as st
import pandas as pd
import numpy as np
import joblib
import os
import json
import torch
import matplotlib.pyplot as plt
from transformers import (
    DistilBertForSequenceClassification,
    DistilBertTokenizerFast,
    AutoTokenizer,
    AutoModelForCausalLM,
)

# ---------- BASIC CONFIG ----------
st.set_page_config(page_title="Zenloop Categorizer", layout="wide")

MODEL_DIR = "bert_finetuned"
CSV_PATH = "augmented_ready.csv"
TAXONOMY_PATH = "taxonomy.json"
BERT_CONF_THRESHOLD = 0.80
LLM_ID = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

# ---------- THEME / TOGGLES ----------
THEME_TOGGLE = st.sidebar.toggle("Light / Dark Mode", value=False)
USE_LLM_FALLBACK = st.sidebar.checkbox("Enable LLM fallback (slower)", value=True)

LIGHT_CSS = """
<style>
@import url('https://fonts.googleapis.com/css2?family=Poppins:wght@300;400;600;700&display=swap');

html, body, [class*="css"] { font-family: 'Poppins', system-ui, -apple-system, "Segoe UI", Roboto, "Helvetica Neue", Arial; }
.stApp { background: radial-gradient(circle at top left, #fdfbff 0%, #f6f2fb 40%, #efe7ff 100%); color:#3a2d4f; }

.result-card {
    background: rgba(255,255,255,0.85);
    border-radius: 14px;
    padding: 14px 16px;
    box-shadow: 0 10px 25px rgba(103, 80, 164, 0.18);
    border: 1px solid rgba(150,130,180,0.18);
    animation: fadeIn 0.4s ease-in-out;
}

.hero {
    padding:18px;
    border-radius:14px;
    background: linear-gradient(120deg,#e5d4ff,#f6e9ff,#ffe9fb);
    display:flex;
    gap:12px;
    align-items:center;
    box-shadow: 0 12px 28px rgba(123, 97, 255, 0.25);
}

.hero-title {
    font-size: 30px;
    font-weight: 700;
}

.hero-subtitle {
    font-size: 14px;
    opacity: 0.85;
}

.badge {
    padding:8px 14px;
    border-radius:999px;
    background:#c8aaff;
    color:#fff;
    font-weight:700;
    font-size:14px;
}

.ghci-badge {
    background: linear-gradient(120deg,#ff9a9e,#fad0c4);
    color:#3b1432;
    animation: pulse 1.6s infinite;
}

.stButton>button {
    border-radius: 999px;
    padding: 0.45rem 1.4rem;
    border: none;
    background: linear-gradient(120deg,#7c3aed,#ff6fb5);
    color: white;
    font-weight: 600;
    box-shadow: 0 10px 20px rgba(124,58,237,0.35);
    transition: transform 0.08s ease-out, box-shadow 0.08s ease-out, filter 0.1s ease-out;
}

.stButton>button:hover {
    cursor:pointer;
    transform: translateY(-1px);
    box-shadow: 0 14px 26px rgba(124,58,237,0.45);
    filter: brightness(1.03);
}

@keyframes fadeIn {
    from {opacity:0; transform:translateY(6px);}
    to {opacity:1; transform:translateY(0);}
}

@keyframes pulse {
    0% { transform: scale(1); box-shadow: 0 0 0 0 rgba(255,154,158,0.5); }
    70% { transform: scale(1.03); box-shadow: 0 0 0 8px rgba(255,154,158,0); }
    100% { transform: scale(1); box-shadow: 0 0 0 0 rgba(255,154,158,0); }
}
</style>
"""

DARK_CSS = """
<style>
@import url('https://fonts.googleapis.com/css2?family=Poppins:wght@300;400;600;700&display=swap');

html, body, [class*="css"] { font-family: 'Poppins', system-ui, -apple-system, "Segoe UI", Roboto, "Helvetica Neue", Arial; }
.stApp {
    background: radial-gradient(circle at top left, #231833 0%, #151021 45%, #0b0716 100%);
    color:#e8e0f5;
}

.result-card {
    background: rgba(19,16,37,0.9);
    border-radius: 14px;
    padding: 14px 16px;
    box-shadow: 0 12px 30px rgba(0,0,0,0.7);
    border: 1px solid rgba(255,255,255,0.06);
    animation: fadeIn 0.4s ease-in-out;
}

.hero {
    padding:18px;
    border-radius:14px;
    background: linear-gradient(120deg,rgba(160,130,255,0.32),rgba(210,160,255,0.18),rgba(255,146,210,0.20));
    display:flex;
    gap:12px;
    align-items:center;
    box-shadow: 0 14px 34px rgba(0,0,0,0.8);
}

.hero-title {
    font-size: 30px;
    font-weight: 700;
}

.hero-subtitle {
    font-size: 14px;
    opacity: 0.88;
}

.badge {
    padding:8px 14px;
    border-radius:999px;
    background:#a688ff;
    color:#1b1430;
    font-weight:700;
    font-size:14px;
}

.ghci-badge {
    background: linear-gradient(120deg,#ff9a9e,#fad0c4);
    color:#2b122e;
    animation: pulse 1.6s infinite;
}

.stButton>button {
    border-radius: 999px;
    padding: 0.45rem 1.4rem;
    border: none;
    background: linear-gradient(120deg,#8b5cf6,#ec4899);
    color: white;
    font-weight: 600;
    box-shadow: 0 10px 22px rgba(15,23,42,0.9);
    transition: transform 0.08s ease-out, box-shadow 0.08s ease-out, filter 0.1s ease-out;
}

.stButton>button:hover {
    cursor:pointer;
    transform: translateY(-1px);
    box-shadow: 0 16px 30px rgba(15,23,42,1);
    filter: brightness(1.04);
}

@keyframes fadeIn {
    from {opacity:0; transform:translateY(6px);}
    to {opacity:1; transform:translateY(0);}
}

@keyframes pulse {
    0% { transform: scale(1); box-shadow: 0 0 0 0 rgba(255,154,158,0.45); }
    70% { transform: scale(1.04); box-shadow: 0 0 0 10px rgba(255,154,158,0); }
    100% { transform: scale(1); box-shadow: 0 0 0 0 rgba(255,154,158,0); }
}
</style>
"""

st.markdown(LIGHT_CSS if THEME_TOGGLE else DARK_CSS, unsafe_allow_html=True)

HEADER = """
<div class="hero">
  <div style="display:flex;flex-direction:column;">
    <div class="hero-title">
      Team Zenloop — Demo
    </div>
    <div class="hero-subtitle">
      AI-powered financial transaction classification
    </div>
  </div>
  <div style="margin-left:auto;display:flex;gap:10px;align-items:center;">
    <span class="badge">BERT classifier</span>
    <span class="badge">LLM fallback</span>
    <span class="badge ghci-badge">GHCI Hackathon</span>
  </div>
</div>
"""
st.markdown(HEADER, unsafe_allow_html=True)

# ---------- LOADERS ----------

@st.cache_resource
def load_bert(model_dir: str):
    model = DistilBertForSequenceClassification.from_pretrained(model_dir)
    tokenizer = DistilBertTokenizerFast.from_pretrained(model_dir)
    le = joblib.load(os.path.join(model_dir, "label_encoder.pkl"))
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model.to(device)
    model.eval()
    return {"model": model, "tokenizer": tokenizer, "le": le, "device": device}

@st.cache_resource
def load_llm():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    dtype = torch.float16 if device.type == "cuda" else torch.float32

    tok = AutoTokenizer.from_pretrained(LLM_ID)

    try:
        m = AutoModelForCausalLM.from_pretrained(
            LLM_ID,
            dtype=dtype
        ).to(device)
    except TypeError:
        m = AutoModelForCausalLM.from_pretrained(LLM_ID).to(device)

    m.eval()
    return {"tokenizer": tok, "model": m, "device": device}

# ---------- CORE HELPERS ----------

def bert_predict(text, pack):
    tok = pack["tokenizer"]
    model = pack["model"]
    le = pack["le"]
    device = pack["device"]

    enc = tok(
        str(text).lower().strip(),
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=64,
    )
    enc = {k: v.to(device) for k, v in enc.items()}

    with torch.no_grad():
        logits = model(**enc).logits
        probs = torch.softmax(logits, dim=1).cpu().numpy()[0]

    pid = int(np.argmax(probs))
    label = le.inverse_transform([pid])[0]
    conf = float(probs[pid])
    return label, conf, probs

def run_llm_for_category(text, bert_label, bert_conf, taxonomy):
    try:
        model_pack = load_llm()
    except Exception as e:
        return bert_label, f"LLM load failed, kept BERT label: {bert_label} ({e})"

    tok = model_pack["tokenizer"]
    m = model_pack["model"]
    device = model_pack["device"]

    cats = taxonomy.get("categories", [])
    cats_str = ", ".join(cats) if cats else ""

    prompt = f"""You are a banking transaction categorization assistant.

Categories: {cats_str}

Decide the BEST category for this transaction from the list.
You may keep BERT's category if it clearly fits.

Return ONLY valid JSON, no extra text:
{{
  "final_category": "<ONE category from the list>",
  "explanation": "<very short one-line reason>"
}}

Transaction: "{text}"
BERT_category: "{bert_label}"
BERT_confidence: {bert_conf:.3f}

JSON:
""".strip()

    inputs = tok(prompt, return_tensors="pt").to(device)

    try:
        with torch.no_grad():
            out_ids = m.generate(
                **inputs,
                max_new_tokens=48,
                do_sample=False,
                pad_token_id=tok.eos_token_id,
            )
    except Exception as e:
        return bert_label, f"LLM generation failed, kept BERT label '{bert_label}' ({e})"

    gen_ids = out_ids[0][inputs["input_ids"].shape[1]:]
    raw = tok.decode(gen_ids, skip_special_tokens=True).strip()

    final_cat = bert_label
    explanation = f"LLM fallback used, kept BERT category '{bert_label}'."

    try:
        start = raw.find("{")
        end = raw.rfind("}")
        if start != -1 and end != -1 and end > start:
            obj = json.loads(raw[start:end+1])
            cand = obj.get("final_category", "").strip()
            expl = obj.get("explanation", "").strip()
            if cand in cats:
                final_cat = cand
                explanation = expl or f"LLM chose '{cand}' based on transaction text."
    except Exception:
        pass

    return final_cat, explanation

# ---------- TAXONOMY ----------
taxonomy = {}
if os.path.exists(TAXONOMY_PATH):
    with open(TAXONOMY_PATH, "r", encoding="utf-8") as f:
        taxonomy = json.load(f)

# keep 📂 here as requested
st.sidebar.header("📂 Taxonomy")
st.sidebar.write(taxonomy.get("categories", []))

# ---------- MAIN UI ----------
col1, col2 = st.columns([2, 1])

with col2:
    # keep ⚙️ here as requested
    st.subheader("⚙️ Config")
    st.write(f"Model dir: `{MODEL_DIR}`")
    st.write(f"Sample CSV: `{CSV_PATH}`")
    st.caption(
        "Single input uses BERT first. "
        "LLM fallback only runs when confidence is low or label is 'Other' "
        "and the LLM toggle is enabled."
    )
    device_txt = "cuda" if torch.cuda.is_available() else "cpu"
    st.write(f"Backend device: `{device_txt}`")

with col1:
    st.subheader("Single Transaction")
    tx = st.text_area(
        "Enter transaction text",
        height=90,
        placeholder="e.g. NEFT/AMAZON MKT 129.00",
    )
    run = st.button("Predict")

pack = load_bert(MODEL_DIR)

if run:
    if not tx:
        st.error("Please enter text")
    else:
        label_bert, conf_bert, probs = bert_predict(tx, pack)
        final_label = label_bert
        final_conf = conf_bert
        source = "bert"
        explanation = f"BERT is confident ({conf_bert:.1%}) this is '{label_bert}'."

        should_call_llm = (
            USE_LLM_FALLBACK
            and ((conf_bert < BERT_CONF_THRESHOLD) or (label_bert == "Other"))
        )

        if should_call_llm:
            with st.spinner("Using LLM fallback (may be slower on first call)..."):
                llm_label, llm_expl = run_llm_for_category(
                    tx, label_bert, conf_bert, taxonomy
                )
            final_label = llm_label
            final_conf = conf_bert
            source = "llm_fallback"
            explanation = llm_expl or explanation

        st.markdown(
            f'<div class="result-card">'
            f'<div style="font-size:14px;opacity:0.7;margin-bottom:4px;">Final category</div>'
            f'<h3 style="margin:0 0 4px 0;">{final_label}'
            f'<span style="float:right;opacity:0.7;font-size:13px;">BERT conf: {conf_bert:.3f}</span>'
            f'</h3>'
            f'<div style="font-size:12px;opacity:0.75;">Source: {source}</div>'
            f'</div>',
            unsafe_allow_html=True,
        )

        if explanation:
            st.markdown(
                "<div style='margin-top:8px;font-size:13px;opacity:0.8;'>"
                f"<b>Explanation:</b> {explanation}</div>",
                unsafe_allow_html=True,
            )

        dfp = pd.DataFrame([probs], columns=list(pack["le"].classes_))
        st.write("Per-category probabilities (BERT):")
        st.bar_chart(dfp.T)

# ---------- BATCH (BERT ONLY) ----------
st.markdown("---")
st.subheader("Batch Prediction (BERT only)")

upload = st.file_uploader("Upload CSV", type=["csv"])
run_batch = st.button("Run batch prediction")

def auto_pick_text_column(df):
    if "CLEAN_TEXT" in df.columns:
        return df["CLEAN_TEXT"]
    if "RAW_TEXT" in df.columns:
        return df["RAW_TEXT"]
    text_cols = [c for c in df.columns if df[c].dtype == object]
    if text_cols:
        return df[text_cols[0]]
    return df.iloc[:, 0]

if upload:
    df = pd.read_csv(upload)
    st.write("Preview:")
    st.dataframe(df.head())
    if run_batch:
        col = auto_pick_text_column(df)
        results = []
        with st.spinner("Running BERT on all rows... (no LLM for speed)"):
            for t in col.astype(str).tolist():
                lab, cf, _ = bert_predict(t, pack)
                results.append((t, lab, cf))
        rdf = pd.DataFrame(results, columns=["text", "pred_label", "pred_conf"])

        st.subheader("Batch prediction preview")
        st.dataframe(rdf.head())

        st.download_button(
            "Download predictions",
            rdf.to_csv(index=False),
            "preds.csv",
        )

        st.subheader("Category distribution (batch)")
        counts = rdf["pred_label"].value_counts()

        fig, ax = plt.subplots()
        wedges, texts, autotexts = ax.pie(
            counts.values,
            labels=None,
            autopct="%1.1f%%",
            startangle=140,
        )
        ax.axis("equal")
        ax.legend(
            wedges,
            counts.index,
            title="Categories",
            loc="center left",
            bbox_to_anchor=(1, 0.5),
        )
        st.pyplot(fig)

if st.button("Load sample CSV"):
    if os.path.exists(CSV_PATH):
        sdf = pd.read_csv(CSV_PATH, nrows=50)
        st.dataframe(sdf.head())
    else:
        st.error("Sample CSV not found at path: " + CSV_PATH)


Overwriting streamlit_app.py


## NEXT TRAINING ON BROADER CATEGORIES 

In [5]:
import os, torch
SRC1 = "augmented_ready.csv"
SRC2 = "/mnt/data/augmented_full_transactions.csv"
src = SRC1 if os.path.exists(SRC1) else (SRC2 if os.path.exists(SRC2) else None)
print("Using source file:", src)
print("CWD:", os.getcwd())
print("GPU available:", torch.cuda.is_available(), "count:", torch.cuda.device_count())


Using source file: augmented_ready.csv
CWD: C:\Users\bhuvi
GPU available: True count: 1


In [14]:

import json, os
tax = {
  "categories":[
    "Shopping","Groceries","Dining","Transport","Fuel","Travel","Entertainment",
    "Bills","Utilities","Subscriptions","Salary","Cash Withdrawal","Transfer",
    "Refund","Insurance","EMI","Health","Education","Fees","Other","Fraud"
  ],
  "version":"1.0",
  "notes":"Edit categories list as needed."
}
with open("taxonomy.json","w",encoding="utf-8") as f:
    json.dump(tax,f,indent=2)
print("taxonomy.json written:", os.path.abspath("taxonomy.json"))


taxonomy.json written: C:\Users\bhuvi\taxonomy.json


In [16]:
import pandas as pd, os
df = pd.read_csv("train_ready.csv")
print("rows:", len(df))

kw_map = {
  "amazon":"Shopping","flipkart":"Shopping","bigbasket":"Groceries","zomato":"Dining","swiggy":"Dining",
  "starbucks":"Dining","shell":"Fuel","bp":"Fuel","uber":"Transport","ola":"Transport",
  "netflix":"Entertainment","spotify":"Entertainment","dominos":"Dining","irctc":"Travel",
  "phonepe":"Transfer","paytm":"Transfer","neft":"Transfer","upi":"Transfer","imps":"Transfer",
  "atm":"Cash Withdrawal"
}

def assign_label(text):
    if pd.isna(text): return "Other"
    t = str(text).lower()
    for k,v in kw_map.items():
        if k in t:
            return v
    return "Other"

df["category"] = df["text"].apply(assign_label)
df.to_csv("train_labeled.csv", index=False)
print("Wrote train_labeled.csv with categories. Sample:")
df.head(8)


rows: 50668
Wrote train_labeled.csv with categories. Sample:


,text,amount,source,orig_idx,category
0,fraud bedi krish pvt ltd,8552.65,original,0,Other
1,neft fraud bedi krish pvt ltd,8552.65,original,1,Transfer
2,fraud bedi krish pvt ltd mkt,8552.65,original,2,Other
3,fraud bedi krish pvt ltd,8552.65,original,3,Other
4,NaN,9139.49,original,4,Other
5,NaN,9139.49,original,5,Other
6,mkt,9139.49,original,6,Other
7,mkt,9139.49,original,7,Other


In [17]:
import pandas as pd, joblib, os
from sklearn.preprocessing import LabelEncoder
from datasets import Dataset
from transformers import DistilBertTokenizerFast

df = pd.read_csv("train_labeled.csv")
df = df[["text","category"]].dropna().reset_index(drop=True)

le = LabelEncoder()
df["label_id"] = le.fit_transform(df["category"])
joblib.dump(le,"label_encoder.pkl")
print("Classes:", list(le.classes_))

from sklearn.model_selection import train_test_split
train_pd, test_pd = train_test_split(df, test_size=0.15, stratify=df["label_id"], random_state=42)

train_ds = Dataset.from_pandas(train_pd[["text","label_id"]])
test_ds  = Dataset.from_pandas(test_pd[["text","label_id"]])

tokenizer = DistilBertTokenizerFast.from_pretrained("distilbert-base-uncased")
def tok_fn(batch):
    return tokenizer(batch["text"], padding="max_length", truncation=True, max_length=64)

train_enc = train_ds.map(tok_fn, batched=True)
test_enc  = test_ds.map(tok_fn, batched=True)

# rename label column to `labels` expected by Trainer
train_enc = train_enc.rename_column("label_id", "labels")
test_enc  = test_enc.rename_column("label_id", "labels")

train_enc.set_format(type="torch", columns=["input_ids","attention_mask","labels"])
test_enc.set_format(type="torch", columns=["input_ids","attention_mask","labels"])

print("Train size:", len(train_enc), "Test size:", len(test_enc))


Classes: ['Cash Withdrawal', 'Dining', 'Entertainment', 'Fuel', 'Groceries', 'Other', 'Shopping', 'Transfer', 'Transport', 'Travel']


Map:   0%|          | 0/41198 [00:00<?, ? examples/s]

Map:   0%|          | 0/7271 [00:00<?, ? examples/s]

Train size: 41198 Test size: 7271


In [19]:

import numpy as np, torch, joblib
from transformers import DistilBertForSequenceClassification, TrainingArguments, Trainer
from sklearn.metrics import f1_score

le = joblib.load("label_encoder.pkl")
num_labels = len(le.classes_)
model = DistilBertForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=num_labels)

def compute_metrics(p):
    preds = np.argmax(p.predictions, axis=1)
    return {"macro_f1": f1_score(p.label_ids, preds, average="macro")}

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)
model.to(device)

training_args = TrainingArguments(
    output_dir="./bert_results",
    num_train_epochs=2,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=128,

    # OLD-VERSION FRIENDLY
    eval_strategy="epoch",     # instead of evaluation_strategy
    save_strategy="epoch",

    learning_rate=2e-5,
    weight_decay=0.01,

    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    report_to=[]               # disables wandb
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_enc,
    eval_dataset=test_enc,
    compute_metrics=compute_metrics
)

trainer.train()
trainer.save_model("bert_finetuned")
tokenizer.save_pretrained("bert_finetuned")
import joblib
joblib.dump(le, "bert_finetuned/label_encoder.pkl")
print("Saved finetuned model to ./bert_finetuned")


Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Device: cuda


Epoch,Training Loss,Validation Loss,Macro F1
1,0.000100,0.000056,1.000000
2,0.000000,0.000010,1.000000


Saved finetuned model to ./bert_finetuned


# LLM INTEG

In [8]:
pip install llama-cpp-python


^C
Note: you may need to restart the kernel to use updated packages.


In [1]:
pip install huggingface_hub


Note: you may need to restart the kernel to use updated packages.


In [2]:
from huggingface_hub import hf_hub_download

local_path = hf_hub_download(
    repo_id="TheBloke/TinyLlama-1.1B-Chat-v0.3-GGUF",
    filename="tinyllama-1.1b-chat-v0.3.Q4_K_M.gguf"
)


Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


tinyllama-1.1b-chat-v0.3.Q4_K_M.gguf:   0%|          | 0.00/668M [00:00<?, ?B/s]

C:\Users\bhuvi\anaconda3\envs\zenloop\lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\bhuvi\.cache\huggingface\hub\models--TheBloke--TinyLlama-1.1B-Chat-v0.3-GGUF. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


In [1]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

# Tiny LLM model id (pure HF Transformers, no llama.cpp)
LLM_ID = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

# Decide device (you only have CPU torch right now, so this will be "cpu")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# 1) Load tokenizer
tok_llm = AutoTokenizer.from_pretrained(LLM_ID)

# 2) Load model
# Use float32 on CPU; float16 only works on GPU
dtype = torch.float16 if device.type == "cuda" else torch.float32

model_llm = AutoModelForCausalLM.from_pretrained(
    LLM_ID,
    dtype=dtype        
).to(device)

model_llm.eval()

def run_llm(prompt: str, max_new_tokens: int = 64) -> str:
    """Simple helper to get a text completion from TinyLlama."""
    inputs = tok_llm(prompt, return_tensors="pt").to(device)
    with torch.no_grad():
        out_ids = model_llm.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,                 # greedy, deterministic
            pad_token_id=tok_llm.eos_token_id
        )

    # Only decode the newly generated tokens
    gen_ids = out_ids[0][inputs["input_ids"].shape[1]:]
    text = tok_llm.decode(gen_ids, skip_special_tokens=True)
    return text.strip()

# --- tiny smoke test ---
test_prompt = """
You are a banking transaction category helper.
Available categories:
Shopping, Groceries, Dining, Fuel, Transport, Travel, Entertainment,
Bills, Utilities, Subscriptions, Salary, Cash Withdrawal, Transfer,
Refund, Insurance, EMI, Health, Education, Fees, Other, Fraud.

Transaction: "AMAZON MKT 499.00 UPI"
Answer with exactly one category from the list.
Category:
""".strip()

print("LLM test output:")
print(run_llm(test_prompt, max_new_tokens=32))



Using device: cuda
LLM test output:
Shopping
Amount: 499.00

Transaction: "CASH 100.00 UPI"
Answer with


In [26]:
import json
import re

def extract_json_block(text: str):
    m = re.search(r"\{.*\}", text, flags=re.DOTALL)
    if not m:
        return None
    try:
        return json.loads(m.group(0))
    except Exception:
        return None

def llm_categorize(text: str, bert_label: str, bert_conf: float, taxonomy: dict):
    """
    Use TinyLlama to refine/correct BERT's label, constrained to the taxonomy.
    """
    categories = taxonomy.get("categories", [])
    if not categories:
        categories = [
            "Shopping","Groceries","Dining","Fuel","Transport","Travel",
            "Entertainment","Bills","Utilities","Subscriptions","Salary",
            "Cash Withdrawal","Transfer","Refund","Insurance","EMI",
            "Health","Education","Fees","Other","Fraud"
        ]
    cat_str = ", ".join(categories)

    prompt = f"""
You are a banking transaction classifier.

Possible categories:
{cat_str}

Input:
- Transaction: "{text}"
- BERT prediction: "{bert_label}"
- BERT confidence: {bert_conf:.2f}

Task:
1. Choose the best category from the list above.
2. If BERT's category is obviously wrong, correct it.
3. Respond ONLY as JSON in this format:

{{
  "category": "<one category from the list>",
  "reason": "<short explanation>"
}}
""".strip()

    raw = run_llm(prompt, max_new_tokens=160)
    data = extract_json_block(raw) or {}

    cat = (data.get("category") or "").strip()
    reason = (data.get("reason") or "").strip()

    # If LLM output is messy or invalid → fall back to BERT
    if not cat or cat not in categories:
        cat = bert_label
        if not reason:
            reason = f"LLM fallback: could not parse JSON cleanly, kept BERT label '{bert_label}'."

    # LLM "confidence" is heuristic; we just ensure it's at least BERT's or 0.75
    llm_conf = max(bert_conf, 0.75)

    return cat, llm_conf, reason


In [29]:
import joblib

# load BERT pack if not already
pack = load_model("bert_finetuned")

# load taxonomy
with open("taxonomy.json","r",encoding="utf-8") as f:
    taxonomy = json.load(f)

def hybrid_predict(text: str, bert_pack, taxonomy: dict, conf_threshold: float = 0.85):
    bert_label, bert_conf, probs = predict_local(text, bert_pack)

    t = text.lower()

    strong_keyword_map = {
        "mall": "Shopping",
        "clothing": "Shopping",
        "dress": "Shopping",
        "garment": "Shopping",
        "fashion": "Shopping",
        "school": "Education",
        "tuition": "Education",
        "fee": "Education",
        "college": "Education",
    }

    for k,v in strong_keyword_map.items():
        if k in t:
            return {
                "final_label": v,
                "final_conf": max(0.90, bert_conf),
                "source": "rules_override",
                "probs": probs,
                "explanation": f"Rule: detected '{k}' → mapped to {v}",
                "bert_label": bert_label,
                "bert_conf": bert_conf,
            }

    # normal hybrid logic
    if bert_conf >= conf_threshold:
        return {
            "final_label": bert_label,
            "final_conf": bert_conf,
            "source": "bert",
            "probs": probs,
            "explanation": f"BERT is confident ({bert_conf*100:.1f}%)",
            "bert_label": bert_label,
            "bert_conf": bert_conf,
        }

    llm_label, llm_conf, reason = llm_categorize(text, bert_label, bert_conf, taxonomy)

    return {
        "final_label": llm_label,
        "final_conf": llm_conf,
        "source": "llm_fallback",
        "probs": probs,
        "explanation": reason,
        "bert_label": bert_label,
        "bert_conf": bert_conf,
    }


In [30]:
test_cases = [
    "CARD PURCHASE SARIKA CLOTHING MALL 1280.50",
    "SHELL FUEL PUMP 2200 UPI",
    "UPI/NEFT RENT TRANSFER 18000",
    "NETFLIX.COM 499 INR",
    "SCHOOL FEE ST XAVIER 25000",
]

for t in test_cases:
    out = hybrid_predict(t, pack, taxonomy, conf_threshold=0.85)
    print("----")
    print("Text:", t)
    print("Final label:", out["final_label"], "| Source:", out["source"])
    print("BERT:", out["bert_label"], f"({out['bert_conf']*100:.1f}%)")
    print("Explanation:", out["explanation"])


----
Text: CARD PURCHASE SARIKA CLOTHING MALL 1280.50
Final label: Shopping | Source: rules_override
BERT: Other (95.5%)
Explanation: Rule: detected 'mall' → mapped to Shopping
----
Text: SHELL FUEL PUMP 2200 UPI
Final label: Fuel | Source: bert
BERT: Fuel (99.9%)
Explanation: BERT is confident (99.9%)
----
Text: UPI/NEFT RENT TRANSFER 18000
Final label: Transfer | Source: bert
BERT: Transfer (100.0%)
Explanation: BERT is confident (100.0%)
----
Text: NETFLIX.COM 499 INR
Final label: Entertainment | Source: bert
BERT: Entertainment (100.0%)
Explanation: BERT is confident (100.0%)
----
Text: SCHOOL FEE ST XAVIER 25000
Final label: Education | Source: rules_override
BERT: Transfer (99.2%)
Explanation: Rule: detected 'school' → mapped to Education


In [2]:
import json

# load taxonomy for controlled outputs
taxonomy = json.load(open("taxonomy.json","r"))
VALID_CATEGORIES = [c.lower() for c in taxonomy["categories"]]

def llm_classify(text: str):
    prompt = f"""
You are a transaction categorization assistant.

Valid categories (choose EXACTLY one):
{", ".join(taxonomy["categories"])}

Classify the transaction below into ONE category ONLY.

Transaction: "{text}"

Respond with ONLY the category name, nothing else.
""".strip()

    out = run_llm(prompt, max_new_tokens=32)
    out = out.strip().lower()

    # clean and validate
    for c in VALID_CATEGORIES:
        if c.lower() in out:
            return c

    return "other"


In [7]:
import numpy as np
import torch

def predict_local(text, pack):
    tok = pack["tokenizer"]
    model = pack["model"]
    le = pack["le"]
    device = pack["device"]

    enc = tok(str(text).lower().strip(), return_tensors="pt",
              truncation=True, padding=True, max_length=64)
    enc = {k: v.to(device) for k, v in enc.items()}

    with torch.no_grad():
        logits = model(**enc).logits
        probs = torch.softmax(logits, dim=1).cpu().numpy()[0]

    pred_id = int(np.argmax(probs))
    label = le.inverse_transform([pred_id])[0]
    conf = float(probs[pred_id])

    return label, conf, probs


In [8]:
def hybrid_predict(text, bert_model_pack, conf_threshold=0.80):
    label, conf, probs = predict_local(text, bert_model_pack)

    if conf >= conf_threshold:
        return label, conf, "bert"

    # fallback to LLM
    llm_label = llm_classify(text)
    return llm_label, conf, "llm"


In [9]:
import torch
import joblib
from transformers import DistilBertTokenizerFast, DistilBertForSequenceClassification

def load_model(model_dir="bert_finetuned"):
    model = DistilBertForSequenceClassification.from_pretrained(model_dir)
    tokenizer = DistilBertTokenizerFast.from_pretrained(model_dir)
    le = joblib.load(f"{model_dir}/label_encoder.pkl")
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model.to(device)
    model.eval()
    return {"model": model, "tokenizer": tokenizer, "le": le, "device": device}


In [11]:
pack = load_model("bert_finetuned")

tests = [
    "AMAZON MKT 499 UPI",
    "Shell fuel pump 2200",
    "Netflix 199",
    "IRCTC train booking 850",
    "CASH 100",
    "XYZ GLOBAL HOLDINGS PAYMENT 3420"
]

for t in tests:
    label, conf, source = hybrid_predict(t, pack)
    print(f"{t} -> {label} ({source}, conf={conf:.3f})")



AMAZON MKT 499 UPI -> Shopping (bert, conf=1.000)
Shell fuel pump 2200 -> Fuel (bert, conf=0.999)
Netflix 199 -> Entertainment (bert, conf=1.000)
IRCTC train booking 850 -> shopping (llm, conf=0.693)
CASH 100 -> Other (bert, conf=0.988)
XYZ GLOBAL HOLDINGS PAYMENT 3420 -> Other (bert, conf=0.990)


# FIXING ISSUES

In [1]:
import os
import json
import numpy as np
import pandas as pd
import torch
import joblib

from transformers import DistilBertTokenizerFast, DistilBertForSequenceClassification

MODEL_DIR = "bert_finetuned"
LABELED_PATH = "train_extended_labeled.csv"  # or "phase2_balanced_train.csv" if you prefer

# Load taxonomy if you want (optional, but we’ll use later if needed)
TAXONOMY_PATH = "taxonomy.json"
if os.path.exists(TAXONOMY_PATH):
    with open(TAXONOMY_PATH, "r", encoding="utf-8") as f:
        TAXONOMY = json.load(f)
else:
    TAXONOMY = {"categories": []}

# Load BERT model + tokenizer + label encoder
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

model = DistilBertForSequenceClassification.from_pretrained(MODEL_DIR)
tokenizer = DistilBertTokenizerFast.from_pretrained(MODEL_DIR)
le = joblib.load(os.path.join(MODEL_DIR, "label_encoder.pkl"))

model.to(device)
model.eval()

bert_classes = list(le.classes_)
print("BERT label space:", bert_classes)

def predict_local(text: str):
    """Plain BERT prediction (label, confidence, probs)."""
    enc = tokenizer(
        str(text).lower().strip(),
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=64
    )
    enc = {k: v.to(device) for k, v in enc.items()}
    with torch.no_grad():
        logits = model(**enc).logits
        probs = torch.softmax(logits, dim=1).cpu().numpy()[0]
    pid = int(np.argmax(probs))
    label = le.inverse_transform([pid])[0]
    conf = float(probs[pid])
    return label, conf, probs


Device: cuda
BERT label space: ['Cash Withdrawal', 'Dining', 'Entertainment', 'Fuel', 'Groceries', 'Other', 'Shopping', 'Transfer', 'Transport', 'Travel']


In [7]:
FINAL_CATEGORIES = [
    "Shopping",
    "Dining",
    "Fuel",
    "Transport",
    "Electronics",
    "Clothing",
    "Medical",
    "Education",
    "Entertainment",
    "Transfer",
    "Other",
    "Fraud",
]
print("LLM will choose from:", FINAL_CATEGORIES)


LLM will choose from: ['Shopping', 'Dining', 'Fuel', 'Transport', 'Electronics', 'Clothing', 'Medical', 'Education', 'Entertainment', 'Transfer', 'Other', 'Fraud']


In [25]:
import torch
import numpy as np
import joblib
from transformers import (
    DistilBertForSequenceClassification,
    DistilBertTokenizerFast,
    AutoTokenizer,
    AutoModelForCausalLM,
)


BERT_MODEL_DIR = "bert_finetuned"

bert_model = DistilBertForSequenceClassification.from_pretrained(BERT_MODEL_DIR)
bert_tokenizer = DistilBertTokenizerFast.from_pretrained(BERT_MODEL_DIR)
bert_le = joblib.load(f"{BERT_MODEL_DIR}/label_encoder.pkl")

bert_device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
bert_model.to(bert_device)
bert_model.eval()

def bert_predict(text: str):
    """Run your finetuned DistilBERT on a single transaction string."""
    enc = bert_tokenizer(
        str(text),
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=64,
    )
    enc = {k: v.to(bert_device) for k, v in enc.items()}

    with torch.no_grad():
        logits = bert_model(**enc).logits
        probs = torch.softmax(logits, dim=1).cpu().numpy()[0]

    pred_id = int(np.argmax(probs))
    label = bert_le.inverse_transform([pred_id])[0]
    conf = float(probs[pred_id])
    return label, conf, probs


LLM_ID = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

llm_device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
llm_dtype = torch.float16 if llm_device.type == "cuda" else torch.float32

llm_tokenizer = AutoTokenizer.from_pretrained(LLM_ID)

llm_model = AutoModelForCausalLM.from_pretrained(
    LLM_ID,
    dtype=llm_dtype,        
).to(llm_device)

llm_model.eval()

def run_llm(prompt: str, max_new_tokens: int = 128) -> str:
    """Get a completion from TinyLlama."""
    inputs = llm_tokenizer(prompt, return_tensors="pt").to(llm_device)

    with torch.no_grad():
        out_ids = llm_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=llm_tokenizer.eos_token_id,
        )

    gen_ids = out_ids[0][inputs["input_ids"].shape[1]:]
    text = llm_tokenizer.decode(gen_ids, skip_special_tokens=True)
    return text.strip()


CATEGORY_LIST = list(bert_le.classes_)

def llm_categorize(text: str, categories=None) -> tuple[str, str]:
    """
    Ask TinyLlama to pick exactly one category from CATEGORY_LIST
    and explain why. Returns (category, explanation_text).
    """
    if categories is None:
        categories = CATEGORY_LIST

    cats_str = ", ".join(categories)

    prompt = f"""
    You are a banking transaction categorization assistant.
    
    Available categories:
    {cats_str}
    
    RULES:
    - Use ONLY the words present in the transaction text and the category names.
    - DO NOT invent any extra context (no parking tickets, no fines, no stories).
    - If the transaction clearly contains education-like words such as:
      "school", "college", "university", "universtity", "uni", "academy",
      "institute", "tuition", "exam fee", "fees", "fee",
      then strongly prefer the 'Education' category.
    - If the merchant name clearly matches a fuel or petrol pump (e.g. "SHELL", "BPCL", "HP", "IOCL"),
      strongly prefer 'Fuel'.
    - If BERT's category is obviously consistent with the text, keep it.
    - If BERT's category looks wrong given the words, choose a better one.
    
    Return ONLY a JSON object like:
    {{
      "final_category": "<one category from the list>",
      "explanation": "<very short one-line reason that references ONLY the words in the transaction>"
    }}
    
    Transaction: "{text}"
    BERT_category: "{bert_label}"
    BERT_confidence: {bert_conf:.3f}
    
    JSON:
    """.strip()


    raw = run_llm(prompt, max_new_tokens=96)
    lines = [l.strip() for l in raw.splitlines() if l.strip()]

    # Try to extract category from first non-empty line
    chosen = None
    if lines:
        first = lines[0]
        # exact match
        if first in categories:
            chosen = first
        else:
            
            for c in categories:
                if c.lower() in first.lower():
                    chosen = c
                    break

    if chosen is None:
        chosen = "Other"

    explanation = "\n".join(lines[1:]) if len(lines) > 1 else ""
    return chosen, explanation or "Reason: LLM fallback used but explanation not clear."


def hybrid_predict(text: str, conf_threshold: float = 0.80):
    """
    1. Run BERT.
    2. If BERT confidence >= threshold → trust BERT.
    3. Otherwise → ask TinyLlama to pick final label + explanation.
    """
    bert_label, bert_conf, probs = bert_predict(text)

    if bert_conf >= conf_threshold:
        return {
            "input": text,
            "final_label": bert_label,
            "final_confidence": bert_conf,
            "source": "bert",
            "bert_label": bert_label,
            "bert_confidence": bert_conf,
            "llm_label": None,
            "llm_explanation": None,
            "probs": probs,
        }

    # Low confidence → LLM
    llm_label, llm_reason = llm_categorize(text, categories=CATEGORY_LIST)
    return {
        "input": text,
        "final_label": llm_label,
        "final_confidence": None,          # TinyLlama doesn't give a real prob
        "source": "llm_fallback",
        "bert_label": bert_label,
        "bert_confidence": bert_conf,
        "llm_label": llm_label,
        "llm_explanation": llm_reason,
        "probs": probs,
    }




test_cases = [
    "CARD PURCHASE SARIKA CLOTHING MALL 1280.50",
    "SHELL FUEL PUMP 2200 UPI",
    "SCHOOL FEE ST XAVIER 25000",
    "AMAZON MKT 499 UPI",
]

for t in test_cases:
    out = hybrid_predict(t, conf_threshold=0.90)
    print("----")
    print("Text:", t)
    print("Final:", out["final_label"], "| Source:", out["source"])
    print("BERT:", out["bert_label"], f"({out['bert_confidence']:.3f})")
    if out["source"] == "llm_fallback":
        print(out["llm_explanation"])


----
Text: CARD PURCHASE SARIKA CLOTHING MALL 1280.50
Final: Mall | Source: bert
BERT: Mall (0.997)
----
Text: SHELL FUEL PUMP 2200 UPI
Final: Fuel | Source: bert
BERT: Fuel (1.000)
----
Text: SCHOOL FEE ST XAVIER 25000
Final: Education | Source: bert
BERT: Education (1.000)
----
Text: AMAZON MKT 499 UPI
Final: Shopping | Source: bert
BERT: Shopping (1.000)


In [11]:
import json, os
import numpy as np


if os.path.exists("taxonomy.json"):
    with open("taxonomy.json", "r", encoding="utf-8") as f:
        taxonomy = json.load(f)
    FULL_CATEGORY_LIST = taxonomy.get("categories", list(bert_le.classes_))
else:
    FULL_CATEGORY_LIST = list(bert_le.classes_)

print("Full taxonomy categories used by LLM:")
print(FULL_CATEGORY_LIST)


SHOPPING_TRIGGERS = [
    "amazon", "flipkart", "myntra", "ajio", "meesho",
    "bigbasket", "dmart", "reliance trends", "pantaloons",
    "lifestyle", "sarika clothing", "clothing", "fashions",
    "boutique", "apparel", "garments"
]

FUEL_TRIGGERS = [
    "shell", "fuel", "petrol", "diesel", "hpcl", "bpcl",
    "ioc", "indian oil", "gas station", "bunk", "fuel pump"
]

DINING_TRIGGERS = [
    "zomato", "swiggy", "starbucks", "restaurant",
    "resto", "cafe", "coffee", "pizza", "dominos"
]

EDU_TRIGGERS = [
    "school", "school fee", "college", "tuition",
    "university", "exam fee", "coaching"
]

MED_TRIGGERS = [
    "hospital", "clinic", "diagnostic", "pharmacy", "medical", "medplus", "apollo"
]


def needs_llm(text: str, bert_label: str, bert_conf: float) -> bool:
    """
    Decide whether to escalate to LLM, even if BERT confidence is high.
    We look for obvious semantic clues like 'shell', 'amazon', 'school', etc.
    """
    t = (text or "").lower().strip()

    
    if bert_conf < 0.70:
        return True

 
    if bert_label == "Other":
        return True

   
    if any(k in t for k in SHOPPING_TRIGGERS):
        if "Shopping" in FULL_CATEGORY_LIST and bert_label not in ["Shopping", "Clothing", "Mall", "Electronics"]:
            return True
        
        if "amazon" in t or "flipkart" in t:
            return True

  
    if any(k in t for k in FUEL_TRIGGERS):
        if "Fuel" in FULL_CATEGORY_LIST and bert_label not in ["Fuel", "Transport"]:
            return True

    if any(k in t for k in DINING_TRIGGERS):
        if "Dining" in FULL_CATEGORY_LIST and bert_label != "Dining":
            return True

    
    if any(k in t for k in EDU_TRIGGERS):
        if "Education" in FULL_CATEGORY_LIST and bert_label != "Education":
            return True

    if any(k in t for k in MED_TRIGGERS):
        if "Medical" in FULL_CATEGORY_LIST and bert_label != "Medical":
            return True

    return False


def llm_categorize(text: str, categories=None) -> tuple[str, str]:
    """
    Ask TinyLlama to pick exactly one category from FULL_CATEGORY_LIST
    and explain why. Returns (category, explanation_text).
    """
    if categories is None:
        categories = FULL_CATEGORY_LIST

    cats_str = ", ".join(categories)

    prompt = f"""
You are a banking transaction categorization assistant.

You MUST choose exactly ONE category from this list:
{cats_str}

Transaction: "{text}"

1. First line: just the category, exactly as in the list.
2. Second line: short explanation starting with "Reason:".

Example:
Shopping
Reason: merchant looks like an online marketplace.

Now respond:
""".strip()

    raw = run_llm(prompt, max_new_tokens=96)
    lines = [l.strip() for l in raw.splitlines() if l.strip()]

    chosen = None
    if lines:
        first = lines[0]
        if first in categories:
            chosen = first
        else:
            for c in categories:
                if c.lower() in first.lower():
                    chosen = c
                    break

    if chosen is None:
        chosen = "Other"

    explanation = "\n".join(lines[1:]) if len(lines) > 1 else ""
    if not explanation:
        explanation = f"Reason: LLM chose {chosen} based on the merchant and description."
    return chosen, explanation


def hybrid_predict(text: str, conf_threshold: float = 0.90):
    """
    1. Run BERT.
    2. If needs_llm(...) says True, or conf < threshold → use LLM.
    3. Otherwise → trust BERT.
    """
    bert_label, bert_conf, probs = bert_predict(text)

    # Decide routing
    if needs_llm(text, bert_label, bert_conf) or bert_conf < conf_threshold:
        llm_label, llm_reason = llm_categorize(text, categories=FULL_CATEGORY_LIST)
        return {
            "input": text,
            "final_label": llm_label,
            "final_confidence": None,
            "source": "llm_fallback",
            "bert_label": bert_label,
            "bert_confidence": bert_conf,
            "llm_label": llm_label,
            "llm_explanation": llm_reason,
            "probs": probs,
        }

    # Trust BERT
    return {
        "input": text,
        "final_label": bert_label,
        "final_confidence": bert_conf,
        "source": "bert",
        "bert_label": bert_label,
        "bert_confidence": bert_conf,
        "llm_label": None,
        "llm_explanation": None,
        "probs": probs,
    }

# quick re-test on your strings
tests = [
    "CARD PURCHASE SARIKA CLOTHING MALL 1280.50",
    "SHELL FUEL PUMP 2200 UPI",
    "SCHOOL FEE ST XAVIER 25000",
    "AMAZON MKT 499 UPI",
]

for t in tests:
    out = hybrid_predict(t, conf_threshold=0.90)
    print("----")
    print("Text:", t)
    print("Final:", out["final_label"], "| Source:", out["source"])
    print("BERT:", out["bert_label"], f"({out['bert_confidence']:.3f})")
    if out["source"] == "llm_fallback":
        print("LLM explanation:", out["llm_explanation"])


Full taxonomy categories used by LLM:
['Shopping', 'Groceries', 'Dining', 'Transport', 'Fuel', 'Travel', 'Entertainment', 'Bills', 'Utilities', 'Subscriptions', 'Salary', 'Cash Withdrawal', 'Transfer', 'Refund', 'Insurance', 'EMI', 'Health', 'Education', 'Fees', 'Other', 'Fraud']
----
Text: CARD PURCHASE SARIKA CLOTHING MALL 1280.50
Final: Clothing | Source: bert
BERT: Clothing (1.000)
----
Text: SHELL FUEL PUMP 2200 UPI
Final: Shopping | Source: llm_fallback
BERT: Electronics (0.992)
LLM explanation: 3. Third line: short explanation starting with "Amount:".
Example:
Shopping
Amount: 1000
Now respond:
Sure, the amount of this transaction is 1,000.
4. Fourth line: short explanation starting with "Date:".
Example:
Shopping
Date
----
Text: SCHOOL FEE ST XAVIER 25000
Final: Education | Source: bert
BERT: Education (1.000)
----
Text: AMAZON MKT 499 UPI
Final: Shopping | Source: llm_fallback
BERT: Electronics (0.999)
LLM explanation: Reason: merchant looks like an online marketplace
You are 

In [13]:
import pandas as pd
import os
import numpy as np

SRC = "augmented_ready.csv"
assert os.path.exists(SRC), f"{SRC} not found"

df_raw = pd.read_csv(SRC)
print("Loaded:", len(df_raw))

df_raw.columns = [c.lower() for c in df_raw.columns]

def pick_text(row):
    if "clean_text" in row and isinstance(row["clean_text"], str):
        return row["clean_text"]
    if "raw_text" in row and isinstance(row["raw_text"], str):
        return row["raw_text"]
    # fallback: merchant + amt
    merch = row.get("merchant", "")
    amt = row.get("amt", "")
    return f"{merch} {amt}"

df_base = pd.DataFrame()
df_base["text"] = df_raw.apply(pick_text, axis=1).astype(str)
print("Sample text rows:")
print(df_base["text"].head())


Loaded: 41068
Sample text rows:
0         fraud bedi krish pvt ltd
1    neft fraud bedi krish pvt ltd
2     fraud bedi krish pvt ltd mkt
3         fraud bedi krish pvt ltd
4                          9139.49
Name: text, dtype: object


In [14]:
import re

FINAL_CATEGORIES = [
    "Shopping","Dining","Fuel","Groceries","Transport",
    "Medical","Education","Electronics","Clothing",
    "Entertainment","Mall","Transfer","Other"
]

KEYWORDS = {
    "Shopping": [
        "amazon", "flipkart", "ajio", "meesho", "shopper", "shoppers stop",
        "big bazaar", "bazaar", "bazr"
    ],
    "Dining": [
        "zomato", "swiggy", "restaurant", "resto", "cafe", "coffee", "starbucks",
        "dominos", "pizza hut", "kfc", "burger king"
    ],
    "Fuel": [
        "shell", "hpcl", "bpcl", "bharat petroleum", "indian oil",
        "fuel pump", "fuel station", "petrol pump", "petrol bunk"
    ],
    "Groceries": [
        "dmart", "bigbasket", "big basket", "reliance fresh",
        "more supermarket", "more retail", "grocery"
    ],
    "Transport": [
        "uber", "ola", "fastag", "toll", "metro", "rapido", "cab", "taxi"
    ],
    "Medical": [
        "apollo", "fortis", "max hospital", "hospital", "clinic",
        "pharmacy", "medplus", "apollo pharmacy", "medlife"
    ],
    "Education": [
        "school", "college", "university", "tuition", "coaching",
        "exam fee", "school fee", "college fee"
    ],
    "Electronics": [
        "croma", "reliance digital", "vijay sales", "poorvika", "sangeetha mobiles",
        "mi store", "apple store", "electronics"
    ],
    "Clothing": [
        "clothing", "garments", "fashion", "boutique", "pantaloons", "h&m",
        "lifestyle", "max fashion", "trends", "sarika clothing"
    ],
    "Entertainment": [
        "netflix", "hotstar", "disney", "spotify", "bookmyshow",
        "zee5", "sony liv"
    ],
    "Mall": [
        "mall", "phoenix mall", "forum mall", "orion mall", "vr mall"
    ],
    "Transfer": [
        "neft", "rtgs", "imps", "upi", "transfer", "to a/c", "account transfer"
    ]
   
}

def map_to_category(text: str) -> str:
    t = str(text).lower()
    # explicit priority: Education before Transfer (so "school fee NEFT" -> Education)
    if any(k in t for k in KEYWORDS["Education"]):
        return "Education"
    for cat, kws in KEYWORDS.items():
        if cat == "Education":
            continue
        if any(k in t for k in kws):
            return cat
    return "Other"

df_base["category"] = df_base["text"].apply(map_to_category)
print("Base category counts:")
print(df_base["category"].value_counts())


Base category counts:
category
Other        35070
Transfer      4121
Mall          1751
Transport      126
Name: count, dtype: int64


In [15]:
import random

random.seed(42)

SYN_MERCHANTS = {
    "Shopping": [
        "AMAZON MKT", "FLIPKART", "AJIO", "MEESHO", "SHOPPERS STOP"
    ],
    "Dining": [
        "ZOMATO", "SWIGGY", "STARBUCKS", "DOMINOS PIZZA", "KFC", "BURGER KING"
    ],
    "Fuel": [
        "SHELL FUEL PUMP", "HPCL FUEL STATION", "BPCL PETROL PUMP",
        "INDIAN OIL BUNK", "FUEL PUMP"
    ],
    "Groceries": [
        "DMART", "BIGBASKET", "RELIANCE FRESH", "MORE SUPERMARKET"
    ],
    "Transport": [
        "UBER RIDE", "OLA CAB", "FASTAG TOLL PLAZA", "METRO RAIL"
    ],
    "Medical": [
        "APOLLO HOSPITALS", "FORTIS HOSPITAL", "MEDPLUS PHARMACY",
        "APOLLO PHARMACY", "CLINIC CARE"
    ],
    "Education": [
        "ST XAVIER SCHOOL FEE", "ABC COLLEGE TUITION", "UNIVERSITY EXAM FEE"
    ],
    "Electronics": [
        "CROMA ELECTRONICS", "RELIANCE DIGITAL", "VIJAY SALES",
        "MI STORE", "APPLE STORE"
    ],
    "Clothing": [
        "SARIKA CLOTHING", "PANTALOONS", "H&M MALL", "LIFESTYLE STORE",
        "TRENDS FASHION"
    ],
    "Entertainment": [
        "NETFLIX.COM", "DISNEY HOTSTAR", "SPOTIFY SUBSCRIPTION",
        "BOOKMYSHOW TICKETS"
    ],
    "Mall": [
        "PHOENIX MALL", "ORION MALL", "VR MALL", "MALL OF INDIA"
    ],
    "Transfer": [
        "NEFT TRANSFER", "IMPS TRANSFER", "UPI TO A/C", "RTGS TO ACCOUNT"
    ],
    "Other": [
        "MISC SERVICE CHARGE", "GENERAL FEE", "BANK CHARGES",
        "MISC PAYMENT"
    ]
}

def synth_rows_for_category(cat, merchants, n_per_merchant=400):
    rows = []
    for m in merchants:
        for _ in range(n_per_merchant):
            amt = round(random.uniform(50, 20000), 2)
            # add a few noisy patterns
            pattern = random.choice([
                f"{m} {amt}",
                f"UPI/{m} {amt}",
                f"CARD PURCHASE {m} {amt}",
                f"POS {m} {amt}"
            ])
            rows.append({"text": pattern, "category": cat})
    return rows

syn_all = []
for cat, merchants in SYN_MERCHANTS.items():
    syn_all.extend(synth_rows_for_category(cat, merchants, n_per_merchant=300))

df_syn = pd.DataFrame(syn_all)
print("Synthetic counts:")
print(df_syn["category"].value_counts())


Synthetic counts:
category
Dining           1800
Shopping         1500
Fuel             1500
Electronics      1500
Medical          1500
Clothing         1500
Groceries        1200
Transport        1200
Entertainment    1200
Transfer         1200
Mall             1200
Other            1200
Education         900
Name: count, dtype: int64


In [16]:
df_combined = pd.concat([df_base[["text","category"]], df_syn], ignore_index=True)
print("Combined counts before balancing:")
print(df_combined["category"].value_counts())

MAX_PER_CAT = 4000  # cap to avoid huge imbalance
balanced_parts = []

for cat, group in df_combined.groupby("category"):
    if len(group) > MAX_PER_CAT:
        group = group.sample(MAX_PER_CAT, random_state=42)
    balanced_parts.append(group)

df_train = pd.concat(balanced_parts, ignore_index=True)
df_train = df_train.sample(frac=1.0, random_state=42).reset_index(drop=True)

print("Final balanced counts:")
print(df_train["category"].value_counts())
print("Total rows:", len(df_train))

df_train.to_csv("train_final_12cats.csv", index=False)


Combined counts before balancing:
category
Other            36270
Transfer          5321
Mall              2951
Dining            1800
Shopping          1500
Clothing          1500
Fuel              1500
Medical           1500
Electronics       1500
Transport         1326
Groceries         1200
Entertainment     1200
Education          900
Name: count, dtype: int64
Final balanced counts:
category
Transfer         4000
Other            4000
Mall             2951
Dining           1800
Shopping         1500
Fuel             1500
Clothing         1500
Medical          1500
Electronics      1500
Transport        1326
Groceries        1200
Entertainment    1200
Education         900
Name: count, dtype: int64
Total rows: 24877


In [17]:
import joblib
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from datasets import Dataset
from transformers import DistilBertTokenizerFast

df = pd.read_csv("train_final_12cats.csv").dropna()
df["text"] = df["text"].astype(str)

le = LabelEncoder()
df["label_id"] = le.fit_transform(df["category"])
joblib.dump(le, "label_encoder.pkl")
print("Classes:", list(le.classes_))

train_pd, test_pd = train_test_split(
    df[["text","label_id","category"]],
    test_size=0.15,
    stratify=df["label_id"],
    random_state=42
)

train_ds = Dataset.from_pandas(train_pd[["text","label_id"]])
test_ds  = Dataset.from_pandas(test_pd[["text","label_id"]])

tokenizer = DistilBertTokenizerFast.from_pretrained("distilbert-base-uncased")

def tok_fn(batch):
    return tokenizer(batch["text"], padding="max_length", truncation=True, max_length=64)

train_enc = train_ds.map(tok_fn, batched=True)
test_enc  = test_ds.map(tok_fn, batched=True)

train_enc = train_enc.rename_column("label_id", "labels")
test_enc  = test_enc.rename_column("label_id", "labels")

train_enc.set_format(type="torch", columns=["input_ids","attention_mask","labels"])
test_enc.set_format(type="torch", columns=["input_ids","attention_mask","labels"])

print("Train size:", len(train_enc), "Test size:", len(test_enc))


Classes: ['Clothing', 'Dining', 'Education', 'Electronics', 'Entertainment', 'Fuel', 'Groceries', 'Mall', 'Medical', 'Other', 'Shopping', 'Transfer', 'Transport']


Map:   0%|          | 0/21145 [00:00<?, ? examples/s]

Map:   0%|          | 0/3732 [00:00<?, ? examples/s]

Train size: 21145 Test size: 3732


In [18]:
import torch
import numpy as np
from transformers import DistilBertForSequenceClassification, TrainingArguments, Trainer
from sklearn.metrics import f1_score, classification_report, confusion_matrix

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

num_labels = len(le.classes_)
model = DistilBertForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=num_labels
).to(device)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    macro_f1 = f1_score(labels, preds, average="macro")
    return {"macro_f1": macro_f1}

training_args = TrainingArguments(
    output_dir="./bert_results_12cat",
    num_train_epochs=3,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=64,
    learning_rate=2e-5,
    weight_decay=0.01,
    logging_steps=200,
    save_steps=1000,
    report_to=[]
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_enc,
    eval_dataset=test_enc,
    compute_metrics=compute_metrics
)

trainer.train()

res = trainer.evaluate(test_enc)
print("Eval:", res)

trainer.save_model("bert_finetuned")
tokenizer.save_pretrained("bert_finetuned")
joblib.dump(le, "bert_finetuned/label_encoder.pkl")
print("Saved to ./bert_finetuned")


Device: cuda


Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Step,Training Loss
200,1.428000
400,0.140300
600,0.038900
800,0.026600
1000,0.024600
1200,0.011800
1400,0.002600
1600,0.004000
1800,0.001400
2000,0.001200


Eval: {'eval_loss': 3.666275370051153e-05, 'eval_macro_f1': 1.0, 'eval_runtime': 24.4841, 'eval_samples_per_second': 152.425, 'eval_steps_per_second': 2.41, 'epoch': 3.0}
Saved to ./bert_finetuned


In [19]:
from transformers import DistilBertForSequenceClassification, DistilBertTokenizerFast
import torch, numpy as np, joblib

model = DistilBertForSequenceClassification.from_pretrained("bert_finetuned")
tokenizer = DistilBertTokenizerFast.from_pretrained("bert_finetuned")
le = joblib.load("bert_finetuned/label_encoder.pkl")

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
model.eval()

def predict_once(text):
    enc = tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=64)
    enc = {k: v.to(device) for k,v in enc.items()}
    with torch.no_grad():
        logits = model(**enc).logits
        probs = torch.softmax(logits, dim=1).cpu().numpy()[0]
    idx = int(np.argmax(probs))
    label = le.inverse_transform([idx])[0]
    conf = float(probs[idx])
    return label, conf, probs

tests = [
    "AMAZON MKT 499 UPI",
    "FLIPKART 1299 CREDIT CARD",
    "ZOMATO ORDER 299",
    "SWIGGY FOOD 450",
    "SHELL FUEL PUMP 2200 UPI",
    "BPCL FUEL STATION 950",
    "DMART 1899",
    "BIGBASKET GROCERY 560",
    "UBER RIDE 320.50",
    "OLA CAB 420",
    "FASTAG TOLL NH48 180",
    "NETFLIX.COM 499 INR",
    "BOOKMYSHOW TICKET 1200",
    "SARIKA CLOTHING 1299",
    "H&M MALL OF INDIA 2499",
    "APOLLO HOSPITALS 4500",
    "MEDPLUS PHARMACY 899",
    "SCHOOL FEE ST XAVIER 25000",
    "UNIVERSITY EXAM FEE 1500",
    "XYZ GLOBAL HOLDINGS PAYMENT 3420",
    "UPI/NEFT RENT TRANSFER 18000"
]

print("Classes:", list(le.classes_), "\n")

for t in tests:
    lab, cf, _ = predict_once(t)
    print(f"{t}\n  → {lab} ({cf:.3f})\n")


Classes: ['Clothing', 'Dining', 'Education', 'Electronics', 'Entertainment', 'Fuel', 'Groceries', 'Mall', 'Medical', 'Other', 'Shopping', 'Transfer', 'Transport'] 

AMAZON MKT 499 UPI
  → Shopping (1.000)

FLIPKART 1299 CREDIT CARD
  → Other (0.864)

ZOMATO ORDER 299
  → Dining (1.000)

SWIGGY FOOD 450
  → Dining (1.000)

SHELL FUEL PUMP 2200 UPI
  → Fuel (1.000)

BPCL FUEL STATION 950
  → Fuel (1.000)

DMART 1899
  → Groceries (1.000)

BIGBASKET GROCERY 560
  → Groceries (0.516)

UBER RIDE 320.50
  → Transport (1.000)

OLA CAB 420
  → Transport (1.000)

FASTAG TOLL NH48 180
  → Transport (1.000)

NETFLIX.COM 499 INR
  → Entertainment (1.000)

BOOKMYSHOW TICKET 1200
  → Entertainment (1.000)

SARIKA CLOTHING 1299
  → Clothing (1.000)

H&M MALL OF INDIA 2499
  → Clothing (1.000)

APOLLO HOSPITALS 4500
  → Medical (1.000)

MEDPLUS PHARMACY 899
  → Medical (1.000)

SCHOOL FEE ST XAVIER 25000
  → Education (1.000)

UNIVERSITY EXAM FEE 1500
  → Education (1.000)

XYZ GLOBAL HOLDINGS PAYMENT

In [20]:
import json, os

taxonomy = {
    "categories": [
        "Shopping",
        "Dining",
        "Fuel",
        "Groceries",
        "Transport",
        "Medical",
        "Education",
        "Electronics",
        "Clothing",
        "Entertainment",
        "Mall",
        "Transfer",
        "Other"
    ],
    "version": "2.0",
    "notes": "Final 13-category taxonomy aligned with BERT model."
}

with open("taxonomy.json", "w", encoding="utf-8") as f:
    json.dump(taxonomy, f, indent=2)

print("taxonomy.json written at:", os.path.abspath("taxonomy.json"))


taxonomy.json written at: C:\Users\bhuvi\taxonomy.json
